# 02 Symbol Optimize - Research Cockpit

Notebook này dùng để tối ưu tham số cho **một symbol** của chiến lược Combo theo quy trình research có kiểm soát. Mục tiêu không phải là tìm một dòng có Sharpe cao nhất trong grid, mà là chọn vùng tham số có edge đủ mẫu, drawdown hợp lý, không quá nhạy với nhiễu, và được xác nhận lại bằng full backtest.

Luồng chính:

1. Khai báo `RUN_CONFIG`: symbol, giai đoạn dữ liệu, vốn, chi phí, ngưỡng lọc và chế độ validate.
2. Audit broker/spec và search space trước khi chạy, để tránh tối ưu trên cấu hình sai.
3. Chạy fast grid search để quét rộng vùng tham số.
4. Lọc candidate theo số lệnh, Profit Factor, Sharpe, drawdown và plateau stability.
5. Full backtest top N candidate, rồi chọn `best_params` từ bảng validate, không chọn trực tiếp từ fast grid.
6. Đọc dashboard cuối: KPI, monthly PnL, equity curve, phân phối lệnh, trade explorer và chart giá.
7. Nếu cần robust hơn: bật walk-forward, Monte Carlo và export bằng chứng research.

Kết quả cuối cùng của notebook là `best_params` và bộ báo cáo validate đi kèm. Sau file này, bước hợp lý là đưa `best_params` sang notebook backtest/portfolio để kiểm tra tương quan, phân bổ vốn và rủi ro danh mục.


## Bản đồ đọc notebook

File này **không thay đổi luật giao dịch Combo**. Nó chỉ điều phối quy trình nghiên cứu để trả lời câu hỏi: với một symbol cụ thể, vùng tham số nào đáng tin nhất để mang sang full backtest, walk-forward và portfolio test?

Cách đọc theo thứ tự:

1. **Config**: xác định symbol, thời gian, vốn, chi phí và tiêu chí pass/fail. Nếu thay đổi nghiên cứu, ưu tiên sửa ở `RUN_CONFIG`.
2. **Spec Audit**: kiểm tra broker specs, spread/swap/lot và search space. Warning không luôn chặn notebook, nhưng là cờ đỏ trước khi dùng kết quả live.
3. **Fast Grid**: quét nhanh nhiều bộ tham số để tìm vùng có tín hiệu. Bảng này dùng để shortlist, không dùng để kết luận cuối.
4. **Filter + Rank**: loại candidate yếu, tính robust score và đánh dấu plateau. Candidate tốt nên vừa có metric ổn vừa không nằm cô lập như spike.
5. **Full Validation**: chạy full backtest cho top N. Đây là bảng quyết định cuối vì nó dùng engine đầy đủ và trade log đầy đủ.
6. **Final Report**: kiểm tra KPI, equity, monthly PnL, distribution và từng trade để hiểu edge đến từ đâu.
7. **Robustness**: walk-forward kiểm tra OOS theo thời gian; Monte Carlo kiểm tra rủi ro do thứ tự lệnh và tail risk.
8. **Export**: lưu lại grid, candidate đã validate và selected params để audit hoặc chuyển sang portfolio layer.

Nguyên tắc đọc nhanh: nếu `grid` đẹp nhưng full validation xấu, bỏ. Nếu full validation đẹp nhưng quá ít lệnh, drawdown tập trung, hoặc plateau yếu, chỉ coi là giả thuyết cần forward test.


In [ ]:
# Cell 1 - Bootstrap import path

import sys
from pathlib import Path


def _find_root(start: Path, marker: str = 'pyproject.toml') -> Path:
    for p in [start, *start.parents]:
        if (p / marker).exists() and (p / 'core_python' / 'shared').exists():
            return p
    raise RuntimeError(f'Cannot find repo root containing {marker!r} and core_python/shared')


ROOT = _find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)

print('ROOT =', ROOT)
print('CORE =', CORE)


In [ ]:
# Cell 2 - Imports
#
# Nhóm import được chia theo vai trò:
# - config/logic: nguồn sự thật của strategy và indicator.
# - symbol.optimize/backtest/walkforward: các runner nghiên cứu một symbol.
# - notebook_utils: các helper hiển thị, filter, rank và validate candidate.
# - monte_carlo: kiểm tra độ nhạy của chuỗi lệnh sau khi đã có trade log thật.
#
# Nếu notebook lỗi ở cell này, thường là do kernel chưa nhận đúng ROOT/CORE ở Cell 1
# hoặc file helper chưa nằm trong sys.path.

import json
from datetime import datetime

import pandas as pd
from IPython.display import display

from shared.monte_carlo import plot_monte_carlo, run_monte_carlo
from strategies.combo.config import (
    SYMBOLS,
    get_indicator_params,
    get_symbol_params,
    get_symbol_search_space,
    summary as strategy_summary,
    validate_config,
)
from strategies.combo.logic import add_combo_indicators
from strategies.combo.notebook_utils import (
    annotate_optimizer_plateau,
    configure_notebook,
    export_result_bundle,
    filter_optimizer_candidates,
    plot_equity_dashboard,
    plot_optimization_dashboard,
    plot_price_with_trades,
    plot_trade_distribution,
    plot_walkforward_dashboard,
    rank_optimizer_candidates,
    show_candidate_validation_report,
    show_kpi_dashboard,
    show_monthly_pnl,
    show_note,
    show_optimizer_filter_report,
    show_run_config,
    show_selected_params,
    show_trade_explorer,
    summarize_candidate_validation,
    summarize_walkforward,
    validate_symbol_candidates,
)
from strategies.combo.symbol.backtest import load_backtest_full, run_symbol_backtest
from strategies.combo.symbol.optimize import run_symbol_grid_search
from strategies.combo.symbol.walkforward import walk_forward_backtest

configure_notebook()
print(strategy_summary())
print('Symbols:', ', '.join(SYMBOLS.keys()))


## Các khái niệm cần nắm trước khi chạy

- **Fast grid**: backtest rút gọn để quét tham số nhanh. Nó hữu ích để loại vùng xấu, nhưng có thể lệch với full engine.
- **Candidate**: một bộ tham số gồm `ktp`, `x`, `ma_period`, `trailing_activation` sau khi được chấm điểm.
- **Filter pass**: candidate vượt ngưỡng tối thiểu về sample size, PF, Sharpe và max drawdown.
- **Robust score**: điểm tổng hợp ưu tiên return/PF/Sharpe nhưng phạt drawdown cao, số lệnh quá ít và spike tham số.
- **Plateau**: dấu hiệu quanh candidate còn nhiều cấu hình lân cận có kết quả ổn. Plateau tốt thường đáng tin hơn một điểm tối ưu cô lập.
- **Full validation**: backtest đầy đủ top candidate. `best_params` phải đến từ bước này.
- **Portfolio step**: tối ưu một symbol chưa đồng nghĩa có thể tăng tỷ trọng trong danh mục. Allocation cần xét correlation, exposure overlap và drawdown đồng thời giữa các symbol.


## 1. Config - khai báo bài toán optimize

Cell kế tiếp là nơi duy nhất nên chỉnh khi đổi bài nghiên cứu. Các nhóm tham số quan trọng:

- `symbol`, `account_mode`, `broker_profile`: xác định thị trường và profile thực thi.
- `date_from`, `date_to`, `max_bars`: xác định mẫu dữ liệu. `date_to=None` nghĩa là dùng đến cuối dữ liệu hiện có.
- `initial_balance`, `costs`: ảnh hưởng trực tiếp đến PnL, DD và lot sizing.
- `min_trades`, `min_profit_factor`, `max_drawdown_pct`, `min_sharpe`: gate tối thiểu trước khi candidate được coi là đáng xem.
- `validate_top_n`: số candidate mang sang full validation. Tăng số này nếu grid rộng hoặc kết quả sát nhau.
- `selection_mode='robust'`: chọn theo điểm bền vững thay vì chỉ một metric đơn lẻ.
- `enable_walkforward`, `enable_monte_carlo`, `export_report`: bật khi cần nghiên cứu sâu hoặc lưu bằng chứng.

Không nên tối ưu quá nhiều chiều cùng lúc nếu sample nhỏ. Mỗi chiều thêm vào làm tăng rủi ro overfitting.


In [ ]:
# Cell 3 - Run config
#
# DOC_CONFIG_FIELDS
# Đọc cell này từ trên xuống dưới: đầu tiên là thị trường cần optimize, sau đó là cửa sổ dữ liệu,
# ngân sách/vốn mô phỏng, tiêu chí lọc candidate, rồi các cờ chạy robustness/export.
# Nếu chỉ đổi symbol hoặc thời gian nghiên cứu, thường chỉ cần chỉnh 3-5 dòng đầu.
#

RUN_CONFIG = {
    'symbol': 'US30',
    'account_mode': 'standard',
    'broker_profile': None,
    'date_from': '2022-01-01',
    'date_to': None,
    'initial_balance': 100_000.0,
    'max_bars': 50_000,
    'top_n': 15,
    'validate_top_n': 10,
    'score_column': 'sharpe',
    'selection_mode': 'robust',
    'min_trades': 40,
    'min_profit_factor': 1.15,
    'max_drawdown_pct': 20.0,
    'min_sharpe': 0.5,
    'search_space_overrides': {},
    'indicator_overrides': {},
    'strategy_overrides': {},
    'costs': {},
    'enable_walkforward': False,
    'walkforward_is_bars': 5_000,
    'walkforward_oos_bars': 1_250,
    'walkforward_step_bars': 1_250,
    'enable_monte_carlo': False,
    'monte_carlo_iter': 1_000,
    'monte_carlo_dd_threshold': 0.20,
    'export_report': False,
}

if RUN_CONFIG['symbol'] not in SYMBOLS:
    raise KeyError(f"Unknown symbol {RUN_CONFIG['symbol']!r}. Available: {list(SYMBOLS)}")

show_run_config('Symbol optimize config', RUN_CONFIG)


## 2. Spec Audit - kiểm tra nền móng trước khi tối ưu

Optimize chỉ có ý nghĩa nếu broker/spec và search space đúng. Cell này kiểm tra config hiện tại và in cảnh báo về những điểm chưa xác minh.

Cần đặc biệt chú ý:

- `spec_verified=False`, spread/swap/commission/contract size chưa chắc đúng.
- Search space thiếu key hoặc quá rộng/quá hẹp.
- Symbol không có trong `SYMBOLS` hoặc mapping dữ liệu sai.

Nếu có error, notebook dừng. Nếu chỉ có warning, vẫn có thể chạy để research, nhưng chưa nên coi kết quả là production-ready.


In [ ]:
# Cell 4 - Broker/spec audit and search space
#
# DOC_AUDIT_PURPOSE
# Audit chạy trước optimize để tránh lỗi nền: broker specs sai, search space thiếu key,
# hoặc config có error. Warning được hiển thị để người đọc biết giả định nào còn yếu.
#

audit = validate_config(broker_profile=RUN_CONFIG.get('broker_profile'))
show_note(
    'Broker/spec audit',
    f"ok={audit['ok']} | warnings={len(audit['warnings'])} | errors={len(audit['errors'])}. "
    'Warnings khong chan optimize, nhung ket qua live chi dang tin khi broker specs da verify.'
)
display(pd.DataFrame({'warnings': audit['warnings'] or ['-']}))
if audit['errors']:
    display(pd.DataFrame({'errors': audit['errors']}))
    raise RuntimeError('Config audit has errors. Fix broker/spec config before optimize.')

search_space = get_symbol_search_space(
    RUN_CONFIG['symbol'],
    broker_profile=RUN_CONFIG.get('broker_profile'),
)
search_space.update(RUN_CONFIG['search_space_overrides'])
required_keys = {'ktp', 'x', 'min_rr'}
missing = sorted(required_keys - set(search_space))
empty = sorted(k for k in required_keys if not search_space.get(k))
if missing or empty:
    raise ValueError(f'Invalid search_space. missing={missing}, empty={empty}')

show_note('Search space', 'Grid search se thu toan bo to hop ben duoi. Luu y grid lon se chay lau.')
display(pd.DataFrame([search_space]).T.rename(columns={0: 'values'}))

## 3. Fast Grid Search - quét rộng để tìm vùng có edge

Cell này chạy `run_symbol_grid_search()` trên toàn bộ search space đã khai báo. Kết quả là bảng `grid`, mỗi dòng là một bộ tham số và metric tương ứng.

Cách đọc:

- `trades`: số lệnh. Quá thấp thì metric dễ nhiễu.
- `profit_factor`: tổng lời / tổng lỗ. Dưới ngưỡng thường không đủ edge sau cost/slippage.
- `max_drawdown_pct`: mức sụt giảm vốn lớn nhất. Cần so với tolerance thực tế.
- `sharpe`, `return_pct`: đo hiệu quả nhưng không nên dùng đơn độc.

Cell này có guard nếu grid rỗng hoặc tất cả candidate không có lệnh. Khi gặp lỗi đó, thường cần kiểm tra date window, dữ liệu, session, spread hoặc điều kiện entry quá chặt.


In [ ]:
# Cell 4b - Optional: X Fill Rate Analysis
#
# Chạy cell này để tìm khoảng x phù hợp theo dữ liệu thực tế của symbol.
# Kết quả là list x_recommended — copy vào search_space_overrides['x'] trong RUN_CONFIG
# nếu muốn dùng thay cho khoảng fallback mặc định.
#
# Bật bằng: RUN_CONFIG['analyze_x_fill_rate'] = True
#

if RUN_CONFIG.get('analyze_x_fill_rate'):
    from strategies.combo.symbol.selection import analyze_x_fill_rate, recommend_x_range
    from strategies.combo.logic import detect_combo_signals, session_mask

    sym_cfg_x = get_symbol_params(RUN_CONFIG['symbol'], broker_profile=RUN_CONFIG.get('broker_profile'))
    raw_x = load_backtest_full(sym_cfg_x['symbol_id'], max_bars=RUN_CONFIG['max_bars'])
    df_x = add_combo_indicators(raw_x.copy(), get_indicator_params())
    mask_x = session_mask(df_x, sym_cfg_x.get('session_hours_utc', []))
    df_x = detect_combo_signals(df_x, mask_x, sym_key=RUN_CONFIG['symbol'], params=get_indicator_params())

    fill_df = analyze_x_fill_rate(RUN_CONFIG['symbol'], df_x)
    x_recommended = recommend_x_range(RUN_CONFIG['symbol'], df_x)

    show_note(
        'X Fill Rate Analysis',
        f"Symbol: {RUN_CONFIG['symbol']} | Signal bars: {int((df_x.get('signal', 0) != 0).sum())} | "
        f"Recommended x: {x_recommended}"
    )
    display(fill_df)
    print(f"\nRecommended x range: {x_recommended}")
    print("→ Để dùng, thêm vào RUN_CONFIG:")
    print(f"  'search_space_overrides': {{'x': {x_recommended}}}")
else:
    print("X fill rate analysis tắt. Set RUN_CONFIG['analyze_x_fill_rate'] = True để chạy.")

In [ ]:
# Cell 5 - Fast grid search
#
# DOC_FAST_GRID_LIMITATION
# Fast grid là vòng quét rộng. Nó giúp tiết kiệm thời gian và tìm vùng tham số đáng chú ý,
# nhưng chưa phải kết luận cuối vì full backtest có execution detail đầy đủ hơn.
#

grid = run_symbol_grid_search(
    RUN_CONFIG['symbol'],
    date_from=RUN_CONFIG['date_from'],
    date_to=RUN_CONFIG['date_to'],
    init_eq=RUN_CONFIG['initial_balance'],
    account_mode=RUN_CONFIG['account_mode'],
    max_bars=RUN_CONFIG['max_bars'],
    indicator_overrides=RUN_CONFIG.get('indicator_overrides') or None,
    strategy_overrides=RUN_CONFIG.get('strategy_overrides') or None,
    costs=RUN_CONFIG.get('costs') or None,
    search_space=search_space,
    broker_profile=RUN_CONFIG.get('broker_profile'),
)

if grid.empty:
    raise RuntimeError('Grid search returned no candidates.')
if 'trades' in grid.columns and int(grid['trades'].sum()) == 0:
    raise RuntimeError('Grid search produced zero trades for all candidates. Check date window, data, session, or search space.')

print('Candidates =', len(grid))
display(grid.head(RUN_CONFIG['top_n']))


## 4. Candidate Filter + Robust Ranking - chọn shortlist có kỷ luật

Bước này biến `grid` thành shortlist có thể validate:

1. Gắn plateau/stability để biết candidate có nằm trong vùng ổn định hay chỉ là điểm spike.
2. Lọc theo rule tối thiểu trong `RUN_CONFIG`.
3. Xếp hạng bằng `robust_score` nếu `selection_mode='robust'`.
4. Nếu không candidate nào pass, notebook fallback sang ranking toàn grid để bạn nhìn được nguyên nhân thất bại, nhưng không nên coi đó là tín hiệu đạt chuẩn.

Một candidate đáng nghiên cứu tiếp nên có đủ lệnh, PF vượt ngưỡng, drawdown không quá cao, Sharpe dương rõ ràng và không quá cô lập so với vùng tham số lân cận.


In [ ]:
# Cell 6 - Filter, plateau and robust ranking
#
# DOC_FILTER_RANK_DECISION
# Rules là cổng chất lượng tối thiểu. Plateau và robust_score là lớp xếp hạng bổ sung để
# ưu tiên candidate ổn định, tránh chọn điểm đẹp nhưng cô lập hoặc sample quá nhỏ.
#

rules = {
    'min_trades': RUN_CONFIG['min_trades'],
    'min_profit_factor': RUN_CONFIG['min_profit_factor'],
    'max_drawdown_pct': RUN_CONFIG['max_drawdown_pct'],
    'min_sharpe': RUN_CONFIG['min_sharpe'],
}

grid_with_plateau = annotate_optimizer_plateau(
    grid,
    score_col=RUN_CONFIG['score_column'],
)
filtered_all = filter_optimizer_candidates(grid_with_plateau, rules, return_all=True)
filtered_ranked = rank_optimizer_candidates(
    filtered_all[filtered_all['filter_pass']],
    mode=RUN_CONFIG['selection_mode'],
)
if filtered_ranked.empty:
    show_note('Fallback', 'Khong candidate nao pass filter. Notebook fallback sang robust ranking toan grid de ban co the xem nguyen nhan.')
    filtered_ranked = rank_optimizer_candidates(filtered_all, mode=RUN_CONFIG['selection_mode'])

show_optimizer_filter_report(filtered_all, title='Candidate filter report')
plot_optimization_dashboard(
    filtered_ranked,
    score_col='robust_score' if 'robust_score' in filtered_ranked.columns else RUN_CONFIG['score_column'],
    top_n=RUN_CONFIG['top_n'],
)
display(filtered_ranked.head(RUN_CONFIG['top_n']))


## 5. Full Validation Top N - quyết định bằng backtest đầy đủ

Fast grid chỉ là vòng loại. Cell này lấy top `validate_top_n` candidate và chạy full backtest cho từng candidate bằng engine chi tiết hơn.

Bảng `validation_compare` giúp so sánh fast vs full:

- Nếu fast đẹp nhưng full xấu: khả năng fast approximation quá lạc quan hoặc candidate nhạy với execution detail.
- Nếu số lệnh lệch mạnh: kiểm tra date range, dữ liệu, logic pending order, trailing hoặc reversal.
- Nếu PF/Sharpe/return giảm nhiều: cần xem cost, slippage, swap và phân phối lệnh.

`best_params` được chọn sau bảng này. Đây là output tham số chính của notebook.


In [ ]:
# Cell 7 - Full backtest validation for top candidates
#
# DOC_VALIDATE_TOP_DECISION
# Chỉ top candidate sau filter/rank mới được chạy full validation để cân bằng tốc độ và độ tin cậy.
# `best_params` ở cuối cell này là tham số nên đọc/ghi nhận, không phải dòng đầu của fast `grid`.
#

top_candidates = filtered_ranked.head(RUN_CONFIG['validate_top_n']).copy()
validation_table, validation_results = validate_symbol_candidates(
    RUN_CONFIG['symbol'],
    top_candidates,
    RUN_CONFIG,
)
validation_compare = summarize_candidate_validation(grid_with_plateau, validation_table)
validation_compare = rank_optimizer_candidates(
    validation_compare,
    mode=RUN_CONFIG['selection_mode'],
)
show_candidate_validation_report(validation_compare)

best = validation_compare.iloc[0]
best_params = {
    'ktp':    float(best['ktp']),
    'x':      float(best['x']),
    'min_rr': float(best['min_rr']),
}
show_selected_params(best_params, title='Final selected params')

## 6. Final Report - đọc kết quả cuối như một trader/risk manager

Cell này chạy lại full backtest cho đúng `best_params` và hiển thị dashboard cuối.

Cách đọc kết quả:

- KPI: kiểm tra return, max DD, PF, Sharpe, win rate, số lệnh và expectancy.
- Monthly PnL: xem lợi nhuận có đến từ vài tháng bất thường hay phân bổ đều qua nhiều regime.
- Equity dashboard: tìm giai đoạn stagnation, cliff drawdown, recovery time.
- Trade distribution: xem tail loss, tail win, skew và độ phụ thuộc vào vài outlier.
- Trade explorer/chart: kiểm tra entry/exit có đúng trực giác market structure hay chỉ là nhiễu.

Nếu báo cáo cuối không đạt tiêu chí risk thực tế, không nên export sang portfolio dù `best_params` đã được chọn.


In [ ]:
# Cell 8 - Final full validation report
#
# DOC_FINAL_REPORT_USAGE
# Cell này tái chạy full backtest với `best_params` để tạo bộ bằng chứng cuối cùng:
# KPI, monthly PnL, equity curve, phân phối lệnh, trade explorer và chart entry/exit.
#

validation = run_symbol_backtest(
    RUN_CONFIG['symbol'],
    init_eq=RUN_CONFIG['initial_balance'],
    account_mode=RUN_CONFIG['account_mode'],
    date_from=RUN_CONFIG['date_from'],
    date_to=RUN_CONFIG['date_to'],
    max_bars=RUN_CONFIG['max_bars'],
    indicator_overrides=RUN_CONFIG.get('indicator_overrides') or None,
    strategy_overrides=RUN_CONFIG.get('strategy_overrides') or None,
    costs=RUN_CONFIG.get('costs') or None,
    broker_profile=RUN_CONFIG.get('broker_profile'),
    symbol_overrides=best_params,
)

show_kpi_dashboard(validation.metrics, title='Final validation KPI')
show_monthly_pnl(validation.metrics, title='Final monthly PnL')
plot_equity_dashboard(validation.equity, validation.trades, title='Final equity dashboard')
plot_trade_distribution(validation.trades)
show_trade_explorer(validation.trades, title='Final trade explorer')
plot_price_with_trades(validation.signal_data, validation.trades, symbol=RUN_CONFIG['symbol'])


## 7. Optional Walk-Forward - kiểm tra out-of-sample theo thời gian

Bật `RUN_CONFIG['enable_walkforward']=True` khi muốn kiểm tra tham số có sống được qua nhiều cửa sổ thị trường hay không.

Ý nghĩa:

- IS window dùng để giả lập giai đoạn nghiên cứu/tối ưu.
- OOS window dùng để giả lập giai đoạn triển khai sau đó.
- Kết quả tốt cần có nhiều OOS window ổn định, không chỉ một cửa sổ thắng lớn.

Nếu OOS liên tục kém hơn IS, đó là dấu hiệu overfitting hoặc chiến lược phụ thuộc regime.


In [ ]:
# Cell 9 - Optional walk-forward robustness
#
# DOC_WALKFORWARD_OPTIONAL
# Walk-forward đang tắt mặc định để notebook chạy nhanh. Bật khi candidate đã qua full validation
# và bạn cần kiểm tra độ bền qua nhiều giai đoạn OOS.
#

if RUN_CONFIG.get('enable_walkforward'):
    sym_cfg = {**get_symbol_params(RUN_CONFIG['symbol'], broker_profile=RUN_CONFIG.get('broker_profile')), **best_params}
    raw = load_backtest_full(
        sym_cfg['symbol_id'],
        max_bars=RUN_CONFIG['max_bars'],
    )
    ind_params = {
        **get_indicator_params(),
        **(RUN_CONFIG.get('indicator_overrides') or {}),
        'MA_PERIOD': 20,
        'KTP': float(best_params['ktp']),
        'X': float(best_params['x']),
    }
    df_ind = add_combo_indicators(raw.copy(), ind_params)
    wf_df, wf_summary = walk_forward_backtest(
        RUN_CONFIG['symbol'],
        df_ind,
        sym_cfg,
        init_eq=RUN_CONFIG['initial_balance'],
        is_bars=RUN_CONFIG['walkforward_is_bars'],
        oos_bars=RUN_CONFIG['walkforward_oos_bars'],
        step_bars=RUN_CONFIG['walkforward_step_bars'],
        strategy=RUN_CONFIG.get('strategy_overrides') or None,
        costs=RUN_CONFIG.get('costs') or None,
        broker_profile=RUN_CONFIG.get('broker_profile'),
    )
    display(pd.DataFrame([wf_summary]).T.rename(columns={0: 'value'}))
    summarize_walkforward(wf_df)
    plot_walkforward_dashboard(wf_df)
else:
    print('Walk-forward disabled. Set RUN_CONFIG[\'enable_walkforward\'] = True to run it.')

## 8. Optional Monte Carlo - kiểm tra rủi ro chuỗi lệnh

Bật `RUN_CONFIG['enable_monte_carlo']=True` sau khi full validation có trade log đủ mẫu.

Monte Carlo ở đây không tạo edge mới. Nó xáo trộn/giả lập chuỗi PnL để ước lượng:

- Xác suất vượt ngưỡng drawdown.
- Khoảng tin cậy của Sharpe.
- Tác động của thứ tự thắng/thua lên equity.

Nếu xác suất vượt DD cao, cần giảm risk per trade, giảm allocation, thêm portfolio cap hoặc bỏ candidate.


In [ ]:
# Cell 10 - Optional Monte Carlo trade-order robustness
#
# DOC_MONTE_CARLO_OPTIONAL
# Monte Carlo dùng trade log của final validation. Nó trả lời câu hỏi risk path:
# nếu thứ tự thắng/thua khác đi, drawdown và Sharpe có còn chấp nhận được không?
#

if RUN_CONFIG.get('enable_monte_carlo'):
    trades_df = pd.DataFrame(validation.trades)
    if trades_df.empty or 'pnl_usd' not in trades_df:
        print('No trade PnL for Monte Carlo.')
    else:
        years = max((validation.equity.index[-1] - validation.equity.index[0]).days / 365.25, 0.1)
        trades_per_year = len(trades_df) / years
        mc = run_monte_carlo(
            trades_df['pnl_usd'].astype(float).tolist(),
            n_iter=RUN_CONFIG['monte_carlo_iter'],
            dd_threshold=RUN_CONFIG['monte_carlo_dd_threshold'],
            initial_balance=RUN_CONFIG['initial_balance'],
            trades_per_year=trades_per_year,
            random_seed=42,
        )
        display(pd.DataFrame({
            'metric': ['prob_exceed_dd', 'sharpe_ci_low', 'sharpe_ci_high', 'trades_per_year'],
            'value': [mc['prob_exceed_dd'], mc['sharpe_ci_low'], mc['sharpe_ci_high'], trades_per_year],
        }))
        plot_monte_carlo(mc)
else:
    print('Monte Carlo disabled. Set RUN_CONFIG[\'enable_monte_carlo\'] = True to run it.')


## 9. Export - lưu bằng chứng research và bước tiếp theo

Bật `RUN_CONFIG['export_report']=True` khi muốn lưu kết quả ra thư mục report.

Các file chính:

- `grid.csv`: toàn bộ fast grid.
- `grid_filtered_ranked.csv`: candidate sau filter/rank.
- `validated_top.csv`: top candidate đã full validation.
- `selected_params.json`: `RUN_CONFIG`, `best_params` và metadata thời điểm chạy.

Bước tiếp theo sau export: đưa `best_params` sang symbol backtest/portfolio notebook, rồi đánh giá correlation, exposure overlap, allocation, max portfolio DD và forward test.


In [ ]:
# Cell 11 - Export bundle
#
# DOC_EXPORT_USAGE
# Export chỉ nên bật khi muốn lưu snapshot nghiên cứu. Các file xuất ra giúp tái kiểm tra
# vì sao một bộ params được chọn và phục vụ bước portfolio/forward test.
#

if RUN_CONFIG.get('export_report'):
    out = export_result_bundle(
        f"{RUN_CONFIG['symbol']}_{RUN_CONFIG['account_mode']}_optimize_research",
        metrics=validation.metrics,
        trades=validation.trades,
        equity=validation.equity,
    )
    grid.to_csv(out / 'grid.csv', index=False, encoding='utf-8-sig')
    filtered_ranked.to_csv(out / 'grid_filtered_ranked.csv', index=False, encoding='utf-8-sig')
    validation_compare.to_csv(out / 'validated_top.csv', index=False, encoding='utf-8-sig')
    metadata = {
        'run_config': RUN_CONFIG,
        'selected_params': best_params,
        'created_at': datetime.utcnow().isoformat() + 'Z',
    }
    with open(out / 'selected_params.json', 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2, default=str)
    print('Exported:', out)
else:
    print("Export disabled. Set RUN_CONFIG['export_report'] = True to save CSV/JSON.")
